In [1]:
import lightgbm as lgb
import numpy as np
from load_data import load_data_with_folds, get_fold_data
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
from pandas.util.version import Infinity


In [2]:
def calculate_tree_scores(booster, shap_values, X_train):
    # Compute mean absolute SHAP values per feature
    total_shap_values = shap_values.sum(axis=0)[:, :-1]
    mean_abs_shap_values = np.mean(np.abs(total_shap_values), axis=0)
    sum_mean_abs_shap_values = np.sum(mean_abs_shap_values)

    prob_shap_values = mean_abs_shap_values / sum_mean_abs_shap_values

    tree_scores = []

    for tree in booster.dump_model()['tree_info']:
        feature_split_count = get_feature_split_count(tree['tree_structure'], X_train)
        tree_score = np.sum(prob_shap_values * feature_split_count)
        tree_scores.append(tree_score)

    tree_scores = np.array(tree_scores)
    sum_scores = np.sum(tree_scores)

    tree_scores = tree_scores / sum_scores

    return tree_scores

def get_feature_split_count(tree_structure, X_train):
    feature_split_count = np.zeros(len(X_train.columns))

    def extract_feature_split_count(tree_structure):
        if 'leaf_index' in tree_structure:
            return
        
        feature_split_count[tree_structure['split_feature']] += 1

        extract_feature_split_count(tree_structure['left_child'])
        extract_feature_split_count(tree_structure['right_child'])

    extract_feature_split_count(tree_structure)

    return feature_split_count

In [3]:
def create_uniqe_seed(drop_seed, fold_idx, iteration):
    if drop_seed is not None:
        return drop_seed

    return fold_idx * 10000 + iteration

def dump_shrinkage(booster):
    tree_info = booster.dump_model()['tree_info']
    shrinkage = []
    for i, tree in enumerate(tree_info):
        shrinkage.append(tree["shrinkage"])

    return shrinkage

In [4]:
def create_shap_drop_callback(X_train, fold_idx, drop_rate, skip_drop, max_drop, use_shap, 
                              shap_values_container, drop_seed=None):
    
    def shap_drop_callback(iteration, userdata):
        booster = userdata
        
        num_trees = booster.num_trees()
        
        # Skip drop if no trees yet
        if num_trees <= 0:
            booster.set_dart_drop_indices([])
            return

        current_shap = shap_values_container[0]

        seed = create_uniqe_seed(drop_seed, fold_idx, iteration)

        # Determine drop indices
        drop_indices = determine_drop_indices(
            booster=booster,
            num_trees=num_trees,
            drop_rate=drop_rate,
            skip_drop=skip_drop,
            max_drop=max_drop,
            X_train=X_train,
            previous_shap_values=current_shap,
            use_shap=use_shap,
            drop_seed=seed
        )

        booster.set_dart_drop_indices(drop_indices)
        
    return shap_drop_callback

def determine_drop_indices(booster, num_trees, drop_rate, skip_drop, max_drop, X_train,
                          previous_shap_values=None, use_shap=False, drop_seed=None):
    rng = np.random.RandomState(drop_seed)
    if rng.rand() < skip_drop:
        return []
    
    effective_drop_rate = drop_rate
    if max_drop > 0 and num_trees > 0:
        effective_drop_rate = min(drop_rate, max_drop / num_trees)

    drop_indices = []

    if use_shap and previous_shap_values is not None:
        tree_scores = calculate_tree_scores(booster, previous_shap_values, X_train)
        
        for i in range(num_trees):
            drop_prob = effective_drop_rate * tree_scores[i] * num_trees
            if rng.rand() < drop_prob:
                drop_indices.append(i)
                if max_drop > 0 and len(drop_indices) >= max_drop:
                    break
        
    else:
        for i in range(num_trees):
            if rng.rand() < effective_drop_rate:
                drop_indices.append(i)

                if max_drop > 0 and len(drop_indices) >= max_drop:
                    break
    
    return drop_indices

def calculate_shap_values_old(booster, X_train):
    model_str = booster.model_to_string()
    snap_booster = lgb.Booster(model_str=model_str)

    n_trees = snap_booster.num_trees()
    shap_prev = 0
    per_tree = []

    for k in range(1, n_trees + 1):
        shap_k = snap_booster.predict(X_train, num_iteration=k, pred_contrib=True)
        per_tree.append(shap_k - shap_prev)
        shap_prev = shap_k

    del snap_booster

    return np.stack(per_tree, axis=0)

def calculate_shap_values_initial(booster, X_train):
    model_str = booster.model_to_string()
    snap = lgb.Booster(model_str=model_str)

    shrinkage = dump_shrinkage(snap)
    n_trees = snap.num_trees()
    
    assert n_trees == 1

    shap_values = snap.predict(
        X_train,
        num_iteration=n_trees,
        pred_contrib=True
    )

    per_tree_shap = shap_values[None, ...]

    del snap
    return per_tree_shap, np.array(shrinkage)

def update_shap_incremental(booster, X_train, per_tree_shap_prev, shrinkage_prev):
    model_str = booster.model_to_string()
    snap = lgb.Booster(model_str=model_str)

    shrinkage_next = np.array(dump_shrinkage(snap))
    n_trees_next = snap.num_trees()
    n_trees_prev = per_tree_shap_prev.shape[0]

    assert n_trees_next > n_trees_prev

    shap_total_next = snap.predict(
        X_train,
        num_iteration=n_trees_next,
        pred_contrib=True
    )

    # Rescale old trees where shrinkage changed
    ratios = shrinkage_next[:n_trees_prev] / shrinkage_prev[:n_trees_prev]
    per_tree_shap_rescaled = per_tree_shap_prev * ratios[:, None, None]

    # New tree SHAP = total - sum(rescaled old)
    sum_old = per_tree_shap_rescaled.sum(axis=0)
    shap_new_tree = shap_total_next - sum_old

    per_tree_shap_next = np.concatenate(
        [per_tree_shap_rescaled, shap_new_tree[None, ...]],
        axis=0
    )

    del snap
    return per_tree_shap_next, shrinkage_next

def compare_shap_values(shap_old, shap_new, rtol=1e-6, atol=1e-8):
    report = {}

    report["same_shape"] = (shap_old.shape == shap_new.shape)
    if not report["same_shape"]:
        report["all_close"] = False
        report["max_abs_diff"] = np.inf
        report["per_tree_all_close"] = []
        report["per_tree_max_abs_diff"] = []
        return report

    n_trees = shap_old.shape[0]

    per_tree_all_close = []
    per_tree_max_abs_diff = []

    for t in range(n_trees):
        diff = shap_old[t] - shap_new[t]
        max_abs = np.max(np.abs(diff))
        ok = np.allclose(shap_old[t], shap_new[t], rtol=rtol, atol=atol)
        per_tree_all_close.append(ok)
        per_tree_max_abs_diff.append(float(max_abs))

    report["per_tree_all_close"] = per_tree_all_close
    report["per_tree_max_abs_diff"] = per_tree_max_abs_diff

    diff_all = shap_old - shap_new
    report["max_abs_diff"] = float(np.max(np.abs(diff_all)))
    report["all_close"] = np.allclose(shap_old, shap_new, rtol=rtol, atol=atol)

    return report

def compare_sample_vs_full(shap_full, shap_sample):
    # Collapse over trees and rows: mean |SHAP| per feature
    mean_abs_full   = np.mean(np.abs(shap_full), axis=(0, 1))   # (n_features,)
    mean_abs_sample = np.mean(np.abs(shap_sample), axis=(0, 1)) # (n_features,)

    # Absolute and relative differences
    abs_diff = mean_abs_sample - mean_abs_full
    rel_diff = abs_diff / (np.abs(mean_abs_full) + 1e-12)

    # Rank correlation of feature importance
    rank_full   = np.argsort(np.argsort(-mean_abs_full))
    rank_sample = np.argsort(np.argsort(-mean_abs_sample))
    spearman, _ = spearmanr(rank_full, rank_sample)

    report = {
        "mean_abs_full": mean_abs_full,
        "mean_abs_sample": mean_abs_sample,
        "abs_diff": abs_diff,
        "rel_diff": rel_diff,
        "max_abs_diff": float(np.max(np.abs(abs_diff))),
        "max_rel_diff": float(np.max(np.abs(rel_diff))),
        "spearman_rank_corr": float(spearman),
    }
    return report
    



In [5]:
def run_single_fold(X, y, patient_ids, fold_indices, fold_idx, params, num_iterations, use_lightgbm_default, use_shap):
    X_train, y_train, X_test, y_test = get_fold_data(
        X, y, patient_ids, fold_indices, fold_idx
    )

    booster = create_booster_with_callback(
        X_train, y_train, X_test, y_test, params, use_lightgbm_default, use_shap, fold_idx=fold_idx
    )

    train_fold_model(booster, num_iterations, use_shap, X_train)

    results = evaluate_fold(booster, X_train, y_train, X_test, y_test, fold_idx)

    return results

def create_booster_with_callback(X_train, y_train, X_test, y_test, params, use_lightgbm_default, use_shap, fold_idx=None):
    train_data = lgb.Dataset(X_train, label=y_train)
    test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

    booster = lgb.Booster(params, train_set=train_data)
    booster.add_valid(test_data, "valid")

    shap_values_container = [None]

    if not use_lightgbm_default:
        callback = create_shap_drop_callback(
            X_train=X_train,
            fold_idx=fold_idx,
            drop_rate=params.get('drop_rate', 0.1),
            skip_drop=params.get('skip_drop', 0.5),
            max_drop=params.get('max_drop', 0),
            use_shap=use_shap,
            shap_values_container=shap_values_container
        )
        booster.set_dart_callback(callback, user_data=booster)

    booster._shap_values_container = shap_values_container

    return booster

def train_fold_model(booster, num_iterations, use_shap, X_train):
    X_shap = X_train.sample(n=100, random_state=42)
    
    shap_values_container = getattr(booster, '_shap_values_container', [None])
    shrinkage = None

    for i in range(num_iterations):
        booster.update()

        if use_shap:
            # shap_values = calculate_shap_values_old(booster, X_train)
            # shap_values_container[0] = shap_values
            if i == 0:
                per_tree_shap, shrinkage = calculate_shap_values_initial(booster, X_shap)
            else:
                per_tree_shap, shrinkage = update_shap_incremental(
                    booster,
                    X_shap,
                    shap_values_container[0],
                    shrinkage
                )

            # create_report(shap_values, per_tree_shap, i)
            # report = compare_sample_vs_full(shap_values, per_tree_shap)
            # print("Spearman rank corr:", report["spearman_rank_corr"])
            # print("Max rel diff:", report["max_rel_diff"])
            # print("Max abs diff:", report["max_abs_diff"])
            
            shap_values_container[0] = per_tree_shap

def evaluate_fold(booster, X_train, y_train, X_test, y_test, fold_idx):
    y_pred_train = booster.predict(X_train)
    y_pred_test = booster.predict(X_test)
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    return {
        'fold_idx': fold_idx,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_samples': len(X_train),
        'test_samples': len(X_test)
    }

def create_report(shap_slow, shap_fast, iteration):
    report = compare_shap_values(shap_slow, shap_fast)

    print(f"Iteration {iteration}")
    print(f"  same_shape: {report['same_shape']}")
    print(f"  all_close:  {report['all_close']}")
    print(f"  max_abs_diff: {report['max_abs_diff']:.3e}")

    # Optional: print per-tree details if something deviates
    if not report["all_close"]:
        for t, (ok, max_d) in enumerate(
            zip(report["per_tree_all_close"], report["per_tree_max_abs_diff"])
        ):
            if not ok:
                print(f"    Tree {t}: max_abs_diff={max_d:.3e}")


In [6]:
def print_cv_summary(fold_results):
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    for result in fold_results:
        print(f"Fold {result['fold_idx']:2d}: Train MSE={result['train_mse']:.4f}, Test MSE={result['test_mse']:.4f} (Train: {result['train_samples']:5d}, Test: {result['test_samples']:5d})")
    avg_test_mse = np.mean([r['test_mse'] for r in fold_results])
    std_test_mse = np.std([r['test_mse'] for r in fold_results])
    print(f"\nAverage Test MSE: {avg_test_mse:.4f} ± {std_test_mse:.4f}")
    return avg_test_mse

In [7]:
def compare_shap_across_iterations(results, rtol=1e-5, atol=1e-8):
    iters = sorted(results.keys())
    comparisons = {}

    for i in range(len(iters) - 1):
        iter_prev = iters[i]
        iter_curr = iters[i + 1]

        sv_prev = results[iter_prev]["shap_values"]
        sv_curr = results[iter_curr]["shap_values"]

        n_trees_prev = sv_prev.shape[0]
        n_trees_curr = sv_curr.shape[0]
        n_common = min(n_trees_prev, n_trees_curr)

        per_tree_equal = []
        for t in range(n_common):
            equal = np.allclose(sv_prev[t], sv_curr[t], rtol=rtol, atol=atol)
            per_tree_equal.append(equal)

        all_equal = all(per_tree_equal)

        comparisons[(iter_prev, iter_curr)] = {
            "per_tree_equal": per_tree_equal,
            "all_equal": all_equal,
        }

    return comparisons

def shap_values_equal(values_1, values_2):
    return np.array_equal(values_1, values_2)

In [8]:
def run_grid_search(base_params, grid, use_lightgbm_default, use_shap, n_folds):
    X, y, patient_ids, fold_indices = load_data_with_folds(
        file_path="../data/slice_localization_data.csv",
        target_col="reference",
        drop_cols=["patientId"],
        n_folds=n_folds,
    )

    all_results = []

    for i, params in enumerate(grid):
        print(f"Running search: {i}/{len(grid)}")
        full_params = {**base_params, **params}

        fold_results = []
        for fold_idx in range(n_folds):
            result = run_single_fold(X, y, patient_ids, fold_indices, fold_idx, full_params, 100, use_lightgbm_default, use_shap)
            fold_results.append(result)
        
        # Store results for this parameter combination
        all_results.append({
            'full_params': full_params,
            'fold_results': fold_results,
        })
    
    return all_results


import json
from datetime import datetime

def save_grid_results(results, output_file, use_lightgbm_default=None, use_shap=None, n_folds=None):
    """Save grid search results to a JSON file."""
    output_data = {
        'timestamp': datetime.now().isoformat(),
        'use_lightgbm_default': use_lightgbm_default,
        'use_shap': use_shap,
        'n_folds': n_folds,
        'results': results
    }
    with open(output_file, 'w') as f:
        json.dump(output_data, f, indent=2)
    print(f"Results saved to {output_file}")
        

In [9]:
from sklearn.model_selection import ParameterGrid

base_params = {
    "objective": "regression",
    "boosting_type": "dart",
    "metric": "mse",
    "verbose": -1,
    "max_depth": 10
}

param_distributions = {
    "learning_rate": [0.1, 0.2, 0.3],
    "num_leaves": [50, 100, 200],
    "feature_fraction_bynode": [0.2, 0.4],
    "drop_rate": [0.025, 0.05, 0.1, 0.2],
}

grid = ParameterGrid(param_distributions)

use_lightgbm_default = True
use_shap = False

grid_results = run_grid_search(base_params, grid, use_lightgbm_default, use_shap, 10)
save_grid_results(grid_results, "../results/grid_search_results_default_100.json", use_lightgbm_default=use_lightgbm_default, use_shap=use_shap, n_folds=10)


Running search: 0/72
Running search: 1/72
Running search: 2/72
Running search: 3/72
Running search: 4/72
Running search: 5/72
Running search: 6/72
Running search: 7/72
Running search: 8/72
Running search: 9/72
Running search: 10/72
Running search: 11/72
Running search: 12/72
Running search: 13/72
Running search: 14/72
Running search: 15/72
Running search: 16/72
Running search: 17/72
Running search: 18/72
Running search: 19/72
Running search: 20/72
Running search: 21/72
Running search: 22/72
Running search: 23/72
Running search: 24/72
Running search: 25/72
Running search: 26/72
Running search: 27/72
Running search: 28/72
Running search: 29/72
Running search: 30/72
Running search: 31/72
Running search: 32/72
Running search: 33/72
Running search: 34/72
Running search: 35/72
Running search: 36/72
Running search: 37/72
Running search: 38/72
Running search: 39/72
Running search: 40/72
Running search: 41/72
Running search: 42/72
Running search: 43/72
Running search: 44/72
Running search: 45/7

In [10]:
# use_lightgbm_default = True
# use_shap = False

# grid_results = run_grid_search(base_params, grid, use_lightgbm_default, use_shap, 10)
# save_grid_results(grid_results, "../results/grid_search_results_default_100.json", use_lightgbm_default=use_lightgbm_default, use_shap=use_shap, n_folds=10)


In [11]:
# use_lightgbm_default = False
# use_shap = True

# grid_results = run_grid_search(base_params, grid, use_lightgbm_default, use_shap, 10)
# save_grid_results(grid_results, "../results/grid_search_results_100.json", use_lightgbm_default=use_lightgbm_default, use_shap=use_shap, n_folds=10)
